# Antibody–Antigen Structural Feature Extraction (fixed, checkpointed)

This notebook rebuilds the ABAG structural feature-extraction workflow with safer checkpoints and corrected CIF identifier matching.

**Main fix:** your ABAG IDs look like `pdb_00001ejo`, while the CIF index may extract only `00001ejo`. This notebook normalizes both formats before matching.

It filters unusable metadata rows, verifies chain availability, maps six CDR sequences, identifies antigen interface residues using a 5 Å side-chain-heavy-atom cutoff, and builds a 20×20 matrix:
- upper triangle: antigen interface internal-contact frequencies
- lower triangle: antibody CDR internal-contact frequencies

It saves progress incrementally. It does not train a CNN or assign PP labels; this notebook is for ABAG structural feature extraction.

## 1. Configuration

Edit only the paths below if needed. `CIF_FOLDER` must point to the folder containing your downloaded/local `.cif` or `.cif.gz` files. The notebook does not require you to upload files interactively.

In [20]:
from pathlib import Path

ABAG_CSV = Path(r"C:\Users\yadavalli charishma\Downloads\splits\splits_final\abag_split.csv")
CIF_FOLDER = Path(r"C:\Users\yadavalli charishma\Downloads\splits\splits_final")

OUTPUT_DIR = Path(r"C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
MATRIX_DIR = OUTPUT_DIR / "matrices"

for folder in (OUTPUT_DIR, CHECKPOINT_DIR, MATRIX_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("ABAG CSV exists:", ABAG_CSV.exists(), ABAG_CSV)
print("CIF folder exists:", CIF_FOLDER.exists(), CIF_FOLDER)
print("Output folder:", OUTPUT_DIR.resolve())

if not ABAG_CSV.exists():
    raise FileNotFoundError(f"ABAG CSV not found: {ABAG_CSV}")
if not CIF_FOLDER.exists():
    raise FileNotFoundError(f"CIF folder not found: {CIF_FOLDER}")

ABAG CSV exists: True C:\Users\yadavalli charishma\Downloads\splits\splits_final\abag_split.csv
CIF folder exists: True C:\Users\yadavalli charishma\Downloads\splits\splits_final
Output folder: C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed


## 2. Imports

In [21]:
import gzip
import json
import re
import shutil
import numpy as np
import pandas as pd
from Bio.PDB import MMCIFParser
from Bio.Align import PairwiseAligner

## 3. Load and filter ABAG metadata

We keep protein/peptide antigens, require antigen-chain information, deduplicate PDB/instance pairs, and require all six CDR sequence fields. These are candidate rows; actual usability is determined by later structural checks. The filtered file is checkpointed.

In [22]:
abag = pd.read_csv(ABAG_CSV, low_memory=False)
print("Input shape:", abag.shape)
print("Columns:", abag.columns.tolist())

required_columns = [
    "PDB_ID", "INSTANCE", "SABDAB_ID", "Hchain", "Lchain",
    "Hseq", "Lseq", "CDRH1", "CDRH2", "CDRH3",
    "CDRL1", "CDRL2", "CDRL3", "agchains", "agtypes",
    "agresolvedseqs", "agexpectedseqs", "method", "resolution",
    "ab_ag_split"
]
missing = [c for c in required_columns if c not in abag.columns]
if missing:
    raise ValueError(f"Required columns missing from abag_split.csv: {missing}")

filtered = abag[
    abag["agtypes"].astype(str).str.contains("PROTEIN|PEPTIDE", case=False, na=False)
].copy()
filtered = filtered[
    filtered["agchains"].notna() &
    filtered["agchains"].astype(str).str.strip().ne("")
].copy()
filtered = filtered.drop_duplicates(subset=["PDB_ID", "INSTANCE"]).copy()

cdr_cols = ["CDRH1", "CDRH2", "CDRH3", "CDRL1", "CDRL2", "CDRL3"]
before_cdr = len(filtered)
filtered = filtered.dropna(subset=cdr_cols).copy()
for col in cdr_cols:
    filtered = filtered[filtered[col].astype(str).str.strip().ne("")].copy()

filtered = filtered[required_columns].reset_index(drop=True)
filtered.to_csv(CHECKPOINT_DIR / "abag_filtered.csv", index=False)

print("Input rows:", len(abag))
print("After antigen type/chain/dedup filters:", before_cdr)
print("After requiring six nonblank CDR sequences:", len(filtered))
print("Saved:", CHECKPOINT_DIR / "abag_filtered.csv")

Input shape: (15641, 48)
Columns: ['INSTANCE', 'PDB_ID', 'SABDAB_ID', 'HEAVY_ID', 'LIGHT_ID', 'PDBdepo', 'SABDABdepo', 'SABDABupdate', 'method', 'resolution', 'type', 'construct', 'species', 'holo', 'Hchain', 'Lchain', 'Hseq', 'Lseq', 'Hseq_expected', 'Lseq_expected', 'VH_numerable_seq', 'VL_numerable_seq', 'VH_numbering_list', 'VL_numbering_list', 'CDRH1', 'CDRH2', 'CDRH3', 'CDRL1', 'CDRL2', 'CDRL3', 'concatCDRseq', 'CDRH123', 'CDRL123', 'agchains', 'agtypes', 'ag_index_list', 'agresolvedseqs', 'agexpectedseqs', 'agresolvedseq_trim_intervals', 'VH_imgt_keep_interval', 'VL_imgt_keep_interval', 'cdrh3_cluster', 'cdrh123_cluster', 'cdrl123_cluster', 'ab_cluster', 'agclusters', 'ab_ag_cluster', 'ab_ag_split']
Input rows: 15641
After antigen type/chain/dedup filters: 7114
After requiring six nonblank CDR sequences: 5432
Saved: C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed\checkpoints\abag_filtered.csv


## 4. Index local CIF structures (corrected identifier matching)

Both `pdb_00001ejo` and `00001ejo` are normalized to the same lookup key. The index includes `.cif` and `.cif.gz` files. If multiple files exist for one ID, chain verification checks each candidate.

In [29]:
def normalize_pdb_id(value):
    """Normalize identifiers such as pdb_00001ejo, 00001ejo, or file stems."""
    s = str(value).strip().lower()
    s = Path(s).name
    s = re.sub(r"\.(cif|mmcif)(\.gz)?$", "", s)
    s = re.sub(r"^pdb_", "", s)
    # For stems like 00001ejo_H_L, keep the identifier before the first underscore.
    s = s.split("_")[0]
    return s

def pdb_id_variants(value):
    raw = str(value).strip().lower()
    norm = normalize_pdb_id(raw)
    return {raw, norm, "pdb_" + norm}

cif_paths = list(CIF_FOLDER.rglob("*.cif")) + list(CIF_FOLDER.rglob("*.cif.gz"))
print("CIF files found recursively:", len(cif_paths))

cif_index = {}
for path in cif_paths:
    # Prefer extracting the ID from the file stem, including common pdb_<id>_... forms.
    stem = path.name.lower()
    stem = re.sub(r"\.cif(\.gz)?$", "", stem)
    norm = normalize_pdb_id(stem)
    if norm:
        cif_index.setdefault(norm, []).append(str(path))

index_path = CHECKPOINT_DIR / "cif_index_fixed.json"
with open(index_path, "w", encoding="utf-8") as f:
    json.dump(cif_index, f, indent=2)

print("Indexed normalized IDs:", len(cif_index))
print("Example index keys:", list(cif_index.keys())[:10])

CIF files found recursively: 15589
Indexed normalized IDs: 8641
Example index keys: ['000010fd', '000010gh', '000010op', '000010or', '000010zo', '000011hk', '000011hw', '000011ol', '000011oo', '000011oq']


## 5. Chain verification (checkpointed)

The previous notebook reported 5,432 rows with `success=False` and `cif_file=NaN`. This section redoes chain verification using normalized identifiers. A previous chain checkpoint is not blindly reused because it may have been generated with the old matching logic.

In [ ]:
parser = MMCIFParser(QUIET=True)

def clean_chain_list(value):
    if pd.isna(value):
        return []
    return [x.strip().upper() for x in re.split(r"[,;|\s]+", str(value).strip()) if x.strip()]

def parse_cif(path, structure_id="structure"):
    path = str(path)
    if path.lower().endswith(".gz"):
        # Bio.PDB parser accepts a file handle; decompress to a temporary cached file.
        temp = CHECKPOINT_DIR / (Path(path).name.replace(".gz", ""))
        if not temp.exists() or temp.stat().st_size == 0:
            with gzip.open(path, "rb") as src, open(temp, "wb") as dst:
                shutil.copyfileobj(src, dst)
        path = str(temp)
    return parser.get_structure(structure_id, path)

def find_matching_cif(pdb_id, required_chains):
    key = normalize_pdb_id(pdb_id)
    required = {str(c).strip().upper() for c in required_chains if str(c).strip()}
    candidates = cif_index.get(key, [])
    failures = []
    for cif_path in candidates:
        try:
            structure = parse_cif(cif_path, key)
            available = {
                str(chain.id).upper()
                for model in structure
                for chain in model
            }
            if required.issubset(available):
                return cif_path, sorted(available), ""
            failures.append(f"{Path(cif_path).name}: required {sorted(required)}, found {sorted(available)}")
        except Exception as exc:
            failures.append(f"{Path(cif_path).name}: parse error {exc}")
    return None, [], "No matching local CIF/required chains. " + " | ".join(failures[:3])

chain_checkpoint = CHECKPOINT_DIR / "chain_verification_fixed.csv"
chain_records = []
for idx, row in filtered.iterrows():
    required = [row["Hchain"], row["Lchain"]] + clean_chain_list(row["agchains"])
    cif_path, available, error = find_matching_cif(row["PDB_ID"], required)
    chain_records.append({
        "row_index": idx,
        "PDB_ID": row["PDB_ID"],
        "INSTANCE": row["INSTANCE"],
        "cif_file": cif_path or "",
        "required_chains": ",".join(sorted({str(c).strip().upper() for c in required if str(c).strip()})),
        "available_chains": ",".join(available),
        "success": bool(cif_path),
        "error": error
    })
    if (idx + 1) % 100 == 0:
        pd.DataFrame(chain_records).to_csv(chain_checkpoint, index=False)
        print(f"Verified {idx + 1}/{len(filtered)} rows")

chain_df = pd.DataFrame(chain_records)
chain_df.to_csv(chain_checkpoint, index=False)
print("Chain verification saved:", chain_checkpoint)
print("Total rows:", len(chain_df))
print("Chain matches:", int(chain_df["success"].sum()))
print("Chain failures:", int((~chain_df["success"]).sum()))
display(chain_df.head())

Verified 100/5432 rows
Verified 200/5432 rows
Verified 300/5432 rows
Verified 400/5432 rows
Verified 500/5432 rows
Verified 600/5432 rows
Verified 700/5432 rows
Verified 800/5432 rows
Verified 900/5432 rows
Verified 1000/5432 rows
Verified 1100/5432 rows
Verified 1200/5432 rows
Verified 1300/5432 rows
Verified 1400/5432 rows
Verified 1500/5432 rows
Verified 1600/5432 rows
Verified 1700/5432 rows
Verified 1800/5432 rows
Verified 1900/5432 rows
Verified 2000/5432 rows
Verified 2100/5432 rows
Verified 2200/5432 rows
Verified 2300/5432 rows
Verified 2400/5432 rows
Verified 2500/5432 rows
Verified 2600/5432 rows
Verified 2700/5432 rows
Verified 2800/5432 rows
Verified 2900/5432 rows
Verified 3000/5432 rows
Verified 3100/5432 rows
Verified 3200/5432 rows
Verified 3300/5432 rows
Verified 3400/5432 rows
Verified 3500/5432 rows
Verified 3600/5432 rows
Verified 3700/5432 rows
Verified 3800/5432 rows
Verified 3900/5432 rows
Verified 4000/5432 rows
Verified 4100/5432 rows
Verified 4200/5432 rows
V

In [10]:
import pandas as pd
from pathlib import Path

p = Path(r"C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed\checkpoints\chain_verification_fixed.csv")

df = pd.read_csv(p)

print("Total saved:", len(df))
print("Successful:", int(df["success"].sum()))
print("Failed:", int((~df["success"]).sum()))

Total saved: 5432
Successful: 3360
Failed: 2072


In [11]:
import pandas as pd
from pathlib import Path

path = Path(
    r"C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed\checkpoints\chain_verification_fixed.csv"
)

df = pd.read_csv(path)

failed = df[df["success"] == False].copy()

print("Total rows:", len(df))
print("Failed rows:", len(failed))

print("\nFailure messages:")
print(
    failed["error"]
    .value_counts()
    .head(15)
    .to_string()
)

Total rows: 5432
Failed rows: 2072

Failure messages:
error
No matching local CIF/required chains. pdb_00001e4w_H_L.cif: required ['B/P', 'H', 'L'], found ['B', 'H', 'L', 'P']                                                                                                                                                                  1
No matching local CIF/required chains. pdb_00002igf_H_L.cif: required ['A/P', 'H', 'L'], found ['A', 'H', 'L', 'P']                                                                                                                                                                  1
No matching local CIF/required chains. pdb_00004hkz_B_A.cif: required ['A', 'B', 'C/H'], found ['A', 'B', 'C', 'H']                                                                                                                                                                  1
No matching local CIF/required chains. pdb_00001bj1_H_L.cif: required ['B/V/W', 'J', 'K'], found ['A', 

In [13]:
import pandas as pd
from pathlib import Path

p = Path(
    r"C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed\checkpoints\chain_verification_fixed.csv"
)

df = pd.read_csv(p)

failed = df[df["success"] == False]

print("Failed rows:", len(failed))
print(failed[["PDB_ID", "INSTANCE", "required_chains", "available_chains"]].head(15).to_string(index=False))

Failed rows: 2072
      PDB_ID         INSTANCE required_chains available_chains
pdb_00001e4w pdb_00001e4w_H_L         B/P,H,L              NaN
pdb_00002igf pdb_00002igf_H_L         A/P,H,L              NaN
pdb_00004hkz pdb_00004hkz_B_A         A,B,C/H              NaN
pdb_00001bj1 pdb_00001bj1_K_J       B/V/W,J,K              NaN
pdb_00001bj1 pdb_00001bj1_H_L       A/V/W,H,L              NaN
pdb_00001i9r pdb_00001i9r_X_Y         C/F,X,Y              NaN
pdb_00001i9r pdb_00001i9r_K_M         B/E,K,M              NaN
pdb_00001i9r pdb_00001i9r_H_L         A/D,H,L              NaN
pdb_00001ken pdb_00001ken_T_U       C/E/J,T,U              NaN
pdb_00001ken pdb_00001ken_H_L       A/C/I,H,L              NaN
pdb_00001a2y pdb_00001a2y_B_A         A,B,C/D              NaN
pdb_00003ehb pdb_00003ehb_C_D        AG/B,C,D              NaN
pdb_00003hb3 pdb_00003hb3_C_D        AG/B,C,D              NaN
pdb_000010or pdb_000010or_U_V         D/F,U,V              NaN
pdb_000010op pdb_000010op_W_X        

In [15]:
import pandas as pd
from pathlib import Path
import re

p = Path(r"C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed\checkpoints\chain_verification_fixed.csv")
df = pd.read_csv(p)

failed = df[df["success"] == False].head(5)

for _, row in failed.iterrows():
    print("PDB:", row["PDB_ID"])
    print("Required chains:", row["required_chains"])
    print("Error:", row["error"])
    print("-" * 40)

PDB: pdb_00001e4w
Required chains: B/P,H,L
Error: No matching local CIF/required chains. pdb_00001e4w_H_L.cif: required ['B/P', 'H', 'L'], found ['B', 'H', 'L', 'P']
----------------------------------------
PDB: pdb_00002igf
Required chains: A/P,H,L
Error: No matching local CIF/required chains. pdb_00002igf_H_L.cif: required ['A/P', 'H', 'L'], found ['A', 'H', 'L', 'P']
----------------------------------------
PDB: pdb_00004hkz
Required chains: A,B,C/H
Error: No matching local CIF/required chains. pdb_00004hkz_B_A.cif: required ['A', 'B', 'C/H'], found ['A', 'B', 'C', 'H']
----------------------------------------
PDB: pdb_00001bj1
Required chains: B/V/W,J,K
Error: No matching local CIF/required chains. pdb_00001bj1_H_L.cif: required ['B/V/W', 'J', 'K'], found ['A', 'H', 'L', 'V', 'W'] | pdb_00001bj1_K_J.cif: required ['B/V/W', 'J', 'K'], found ['B', 'J', 'K', 'V', 'W']
----------------------------------------
PDB: pdb_00001bj1
Required chains: A/V/W,H,L
Error: No matching local CIF/req

In [16]:
import re

def clean_chain_list(value):
    if pd.isna(value):
        return []
    return [
        x.strip().upper()
        for x in re.split(r"[,;/|\s]+", str(value).strip())
        if x.strip()
    ]

# Test the chain parsing only — no CIF files are processed.
tests = ["B/P,H,L", "A/P,H,L", "A,B,C/H", "B/V/W,J,K"]

for value in tests:
    print(value, "->", clean_chain_list(value))

B/P,H,L -> ['B', 'P', 'H', 'L']
A/P,H,L -> ['A', 'P', 'H', 'L']
A,B,C/H -> ['A', 'B', 'C', 'H']
B/V/W,J,K -> ['B', 'V', 'W', 'J', 'K']


In [32]:
# Paths for retrying failed chain verification records
from pathlib import Path

OUTPUT_DIR = Path(
    r"C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed"
)

CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"

original_checkpoint = CHECKPOINT_DIR / "chain_verification_fixed.csv"
retry_checkpoint = CHECKPOINT_DIR / "chain_verification_retry.csv"

print("Original checkpoint exists:", original_checkpoint.exists())
print("Retry checkpoint:", retry_checkpoint)

Original checkpoint exists: True
Retry checkpoint: C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed\checkpoints\chain_verification_retry.csv


In [34]:
# Retry failed rows only; preserve the original checkpoint.

import pandas as pd
import re
from pathlib import Path

original = CHECKPOINT_DIR / "chain_verification_fixed.csv"
retry_file = CHECKPOINT_DIR / "chain_verification_retry.csv"

chain_df = pd.read_csv(original)

failed_rows = chain_df[~chain_df["success"].fillna(False)].copy()

print("Rows to retry:", len(failed_rows))

retry_records = []

for n, (_, old) in enumerate(failed_rows.iterrows(), start=1):
    # Recover the corresponding input row using its saved row_index.
    idx = int(old["row_index"])
    row = filtered.loc[idx]

    required = (
        [row["Hchain"], row["Lchain"]]
        + clean_chain_list(row["agchains"])
    )

    cif_path, available, error = find_matching_cif(
        row["PDB_ID"], required
    )

    retry_records.append({
        "row_index": idx,
        "PDB_ID": row["PDB_ID"],
        "INSTANCE": row["INSTANCE"],
        "cif_file": cif_path or "",
        "required_chains": ",".join(
            sorted({
                c.strip().upper()
                for c in required
                if c and str(c).strip()
            })
        ),
        "available_chains": ",".join(available),
        "success": bool(cif_path),
        "error": error
    })

    if n % 25 == 0:
        pd.DataFrame(retry_records).to_csv(retry_file, index=False)
        print(f"Retried {n}/{len(failed_rows)} failed rows")

# Save final retry results separately.
pd.DataFrame(retry_records).to_csv(retry_file, index=False)

print("\nRetry results saved to:", retry_file)
print("Rows retried:", len(retry_records))
print("Recovered:", sum(r["success"] for r in retry_records))
print("Still failing:", sum(not r["success"] for r in retry_records))

Rows to retry: 2072


NameError: name 'find_matching_cif' is not defined

In [40]:
import pandas as pd
from pathlib import Path

# Paths
CHECKPOINT_DIR = Path(
    r"C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed\checkpoints"
)

original_file = CHECKPOINT_DIR / "chain_verification_fixed.csv"
retry_file = CHECKPOINT_DIR / "chain_verification_retry.csv"
merged_file = CHECKPOINT_DIR / "chain_verification_merged.csv"

# Safety checks: required notebook variables/functions must already exist.
required_names = ["filtered", "cif_index", "clean_chain_list", "find_matching_cif"]
missing = [name for name in required_names if name not in globals()]

if missing:
    raise RuntimeError(
        f"Missing notebook variables/functions: {missing}. "
        "Run the cells that load filtered data, index CIF files, "
        "and define the chain-verification helpers first."
    )

if not original_file.exists():
    raise FileNotFoundError(f"Original checkpoint not found: {original_file}")

# Load original checkpoint without modifying it.
original = pd.read_csv(original_file)
original["success"] = (
    original["success"].astype(str).str.strip().str.lower()
    .isin(["true", "1", "yes"])
)

failed = original.loc[~original["success"]].copy()

# Resume from any retry progress already saved.
columns = [
    "row_index", "PDB_ID", "INSTANCE", "cif_file",
    "required_chains", "available_chains", "success", "error"
]

if retry_file.exists() and retry_file.stat().st_size > 0:
    previous = pd.read_csv(retry_file)
    if not set(columns).issubset(previous.columns):
        raise ValueError(
            "Existing retry checkpoint has unexpected columns. "
            "It has been left untouched; inspect it before proceeding."
        )
    previous = previous[columns].drop_duplicates("row_index", keep="last")
else:
    previous = pd.DataFrame(columns=columns)

completed = set(
    pd.to_numeric(previous["row_index"], errors="coerce")
    .dropna().astype(int)
)

pending = failed[
    ~pd.to_numeric(failed["row_index"], errors="coerce")
      .fillna(-1).astype(int).isin(completed)
].copy()

records = previous.to_dict("records")

print("Failed rows in original checkpoint:", len(failed))
print("Already retried:", len(completed))
print("Rows remaining to retry:", len(pending))

for n, (_, old) in enumerate(pending.iterrows(), start=1):
    idx = int(old["row_index"])

    try:
        if idx not in filtered.index:
            raise IndexError(
                f"Row index {idx} is absent from filtered data."
            )

        row = filtered.loc[idx]

        required = (
            [row["Hchain"], row["Lchain"]]
            + clean_chain_list(row["agchains"])
        )
        required = sorted({
            str(c).strip().upper()
            for c in required
            if pd.notna(c) and str(c).strip()
        })

        cif_path, available, error = find_matching_cif(
            row["PDB_ID"], required
        )

        record = {
            "row_index": idx,
            "PDB_ID": row["PDB_ID"],
            "INSTANCE": row["INSTANCE"],
            "cif_file": cif_path or "",
            "required_chains": ",".join(required),
            "available_chains": ",".join(available),
            "success": bool(cif_path),
            "error": error
        }

    except Exception as exc:
        record = {
            "row_index": idx,
            "PDB_ID": old.get("PDB_ID", ""),
            "INSTANCE": old.get("INSTANCE", ""),
            "cif_file": "",
            "required_chains": "",
            "available_chains": "",
            "success": False,
            "error": str(exc)
        }

    records.append(record)

    if n % 25 == 0:
        pd.DataFrame(records, columns=columns).drop_duplicates(
            "row_index", keep="last"
        ).to_csv(retry_file, index=False)
        print(f"Retried {n}/{len(pending)}; progress saved.")

# Final retry checkpoint
retry_df = pd.DataFrame(records, columns=columns).drop_duplicates(
    "row_index", keep="last"
)
retry_df.to_csv(retry_file, index=False)

# Merge only successful recoveries; preserve original failures otherwise.
merged = original.copy()
merged_by_index = merged.set_index("row_index", drop=False)

for _, rr in retry_df.iterrows():
    idx = int(rr["row_index"])
    if bool(rr["success"]) and idx in merged_by_index.index:
        for col in ["cif_file", "success", "error"]:
            if col in merged_by_index.columns:
                merged_by_index.at[idx, col] = rr[col]

merged = merged_by_index.reset_index(drop=True)
merged.to_csv(merged_file, index=False)

print("\nRETRY FINISHED")
print("Retry checkpoint:", retry_file)
print("Merged checkpoint:", merged_file)
print("Successful retry records:", int(retry_df["success"].sum()))
print("Original checkpoint preserved:", original_file.exists())

Failed rows in original checkpoint: 2072
Already retried: 0
Rows remaining to retry: 2072
Retried 25/2072; progress saved.
Retried 50/2072; progress saved.
Retried 75/2072; progress saved.
Retried 100/2072; progress saved.
Retried 125/2072; progress saved.
Retried 150/2072; progress saved.
Retried 175/2072; progress saved.
Retried 200/2072; progress saved.
Retried 225/2072; progress saved.
Retried 250/2072; progress saved.
Retried 275/2072; progress saved.
Retried 300/2072; progress saved.
Retried 325/2072; progress saved.
Retried 350/2072; progress saved.
Retried 375/2072; progress saved.
Retried 400/2072; progress saved.
Retried 425/2072; progress saved.
Retried 450/2072; progress saved.
Retried 475/2072; progress saved.
Retried 500/2072; progress saved.
Retried 525/2072; progress saved.
Retried 550/2072; progress saved.
Retried 575/2072; progress saved.
Retried 600/2072; progress saved.
Retried 625/2072; progress saved.
Retried 650/2072; progress saved.
Retried 675/2072; progress sa

In [38]:

print("filtered exists:", "filtered" in globals())
print("cif_index exists:", "cif_index" in globals())
print("clean_chain_list exists:", "clean_chain_list" in globals())
print("parse_cif exists:", "parse_cif" in globals())
print("find_matching_cif exists:", "find_matching_cif" in globals())


filtered exists: True
cif_index exists: True
clean_chain_list exists: True
parse_cif exists: False
find_matching_cif exists: False


In [39]:
import gzip
import shutil
from pathlib import Path
from Bio.PDB import MMCIFParser

parser = MMCIFParser(QUIET=True)

def parse_cif(path, structure_id="structure"):
    path = str(path)

    if path.lower().endswith(".gz"):
        temp = CHECKPOINT_DIR / Path(path).name[:-3]

        if not temp.exists() or temp.stat().st_size == 0:
            with gzip.open(path, "rb") as src, open(temp, "wb") as dst:
                shutil.copyfileobj(src, dst)

        path = str(temp)

    return parser.get_structure(structure_id, path)


def find_matching_cif(pdb_id, required_chains):
    key = normalize_pdb_id(pdb_id)

    required = {
        str(c).strip().upper()
        for c in required_chains
        if str(c).strip()
    }

    candidates = cif_index.get(key, [])
    failures = []

    for cif_path in candidates:
        try:
            structure = parse_cif(cif_path, key)

            available = {
                str(chain.id).upper()
                for model in structure
                for chain in model
            }

            if required.issubset(available):
                return cif_path, sorted(available), ""

            failures.append(
                f"{Path(cif_path).name}: "
                f"required {sorted(required)}, found {sorted(available)}"
            )

        except Exception as exc:
            failures.append(f"{Path(cif_path).name}: parse error {exc}")

    return (
        None,
        [],
        "No matching local CIF/required chains. " + " | ".join(failures[:3])
    )

print("parse_cif defined:", callable(parse_cif))
print("find_matching_cif defined:", callable(find_matching_cif))

parse_cif defined: True
find_matching_cif defined: True


## 6. CDR sequence mapping helpers

Uses exact sequence matching first, then a conservative local alignment fallback to tolerate small structure/metadata sequence differences. A fallback mapping is accepted only when identity and coverage thresholds pass. Unmapped CDRs are reported rather than guessed.

In [35]:
AA3_TO_1 = {
    "ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C",
    "GLN":"Q","GLU":"E","GLY":"G","HIS":"H","ILE":"I",
    "LEU":"L","LYS":"K","MET":"M","PHE":"F","PRO":"P",
    "SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V"
}
STANDARD_AA = set(AA3_TO_1.values())

def normalize_sequence(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^A-Z]", "", str(value).upper())

def get_chain_residues(structure, chain_id):
    chain_id = str(chain_id).strip().upper()
    for model in structure:
        for chain in model:
            if str(chain.id).upper() == chain_id:
                residues = []
                for residue in chain:
                    aa = AA3_TO_1.get(residue.get_resname().upper())
                    if aa in STANDARD_AA:
                        residues.append(residue)
                return residues
    return []

def map_sequence_to_residues(cdr_sequence, residues):
    target = normalize_sequence(cdr_sequence)
    if not target:
        return None, "empty CDR sequence"
    chain_seq = "".join(AA3_TO_1.get(r.get_resname().upper(), "X") for r in residues)
    start = chain_seq.find(target)
    if start >= 0:
        return residues[start:start+len(target)], "exact"
    # Local alignment fallback; only accept if most of the target is aligned with high identity.
    if not residues:
        return None, "empty chain"
    aligner = PairwiseAligner()
    aligner.mode = "local"
    aligner.match_score = 2
    aligner.mismatch_score = -1
    aligner.open_gap_score = -5
    aligner.extend_gap_score = -0.5
    alignment = aligner.align(chain_seq, target)[0]
    coords = alignment.aligned
    if len(coords[0]) == 0:
        return None, "no alignment"
    # Map the longest aligned block from target to chain coordinates.
    best = max(zip(coords[0], coords[1]), key=lambda pair: min(pair[0][1]-pair[0][0], pair[1][1]-pair[1][0]))
    (s0, s1), (t0, t1) = best
    aligned_len = min(s1-s0, t1-t0)
    coverage = aligned_len / max(1, len(target))
    aligned_target = target[t0:t1]
    aligned_chain = chain_seq[s0:s1]
    identity = sum(a == b for a, b in zip(aligned_chain, aligned_target)) / max(1, min(len(aligned_chain), len(aligned_target)))
    if coverage >= 0.8 and identity >= 0.8:
        return residues[s0:s1], f"alignment coverage={coverage:.2f}, identity={identity:.2f}"
    return None, f"sequence not mapped (coverage={coverage:.2f}, identity={identity:.2f})"

def map_all_cdrs(row, structure):
    H = get_chain_residues(structure, row["Hchain"])
    L = get_chain_residues(structure, row["Lchain"])
    cdr_map, methods = {}, {}
    for name, col, residues in [
        ("H1","CDRH1",H), ("H2","CDRH2",H), ("H3","CDRH3",H),
        ("L1","CDRL1",L), ("L2","CDRL2",L), ("L3","CDRL3",L)
    ]:
        mapped, method = map_sequence_to_residues(row[col], residues)
        if mapped is None:
            return None, f"{name}: {method}", methods
        cdr_map[name] = mapped
        methods[name] = method
    return cdr_map, None, methods

## 7. CDR mapping checkpoint (resumable)

This checkpoint stores mapping status, not the residue objects themselves. Each successful row is mapped again when its matrix is built; the checkpoint prevents repeating the initial diagnostic mapping stage. Empty or malformed checkpoints are rebuilt safely.

In [25]:
cdr_checkpoint = CHECKPOINT_DIR / "cdr_mapping_fixed.csv"
cdr_columns = ["row_index","PDB_ID","INSTANCE","cif_file","success","error","mapping_methods"]

old = None
if cdr_checkpoint.exists() and cdr_checkpoint.stat().st_size > 0:
    try:
        candidate = pd.read_csv(cdr_checkpoint)
        if all(c in candidate.columns for c in cdr_columns):
            old = candidate
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        old = None

if old is None:
    cdr_df = pd.DataFrame(columns=cdr_columns)
else:
    cdr_df = old.copy()

completed = set(pd.to_numeric(cdr_df.get("row_index", pd.Series(dtype=int)), errors="coerce").dropna().astype(int))
pending = chain_df[chain_df["success"]].copy()
pending = pending[~pending["row_index"].astype(int).isin(completed)]

records = cdr_df.to_dict("records")
print("Chain-valid rows:", int(chain_df["success"].sum()))
print("Previously mapped rows:", len(completed))
print("Rows remaining for CDR mapping:", len(pending))

for n, (_, chain_row) in enumerate(pending.iterrows(), start=1):
    idx = int(chain_row["row_index"])
    row = filtered.loc[idx]
    try:
        structure = parse_cif(chain_row["cif_file"], str(row["PDB_ID"]))
        cdr_map, error, methods = map_all_cdrs(row, structure)
        rec = {
            "row_index": idx, "PDB_ID": row["PDB_ID"], "INSTANCE": row["INSTANCE"],
            "cif_file": chain_row["cif_file"], "success": cdr_map is not None,
            "error": error or "", "mapping_methods": json.dumps(methods)
        }
    except Exception as exc:
        rec = {
            "row_index": idx, "PDB_ID": row["PDB_ID"], "INSTANCE": row["INSTANCE"],
            "cif_file": chain_row["cif_file"], "success": False,
            "error": str(exc), "mapping_methods": "{}"
        }
    records.append(rec)
    if n % 50 == 0:
        pd.DataFrame(records, columns=cdr_columns).to_csv(cdr_checkpoint, index=False)
        print(f"Mapped {n}/{len(pending)} pending rows")

cdr_df = pd.DataFrame(records, columns=cdr_columns).drop_duplicates("row_index", keep="last")
cdr_df.to_csv(cdr_checkpoint, index=False)
print("CDR mapping checkpoint:", cdr_checkpoint)
print("CDR mapping success:", int(cdr_df["success"].astype(bool).sum()))
print("CDR mapping failed:", int((~cdr_df["success"].astype(bool)).sum()))
display(cdr_df.head())

Chain-valid rows: 3360
Previously mapped rows: 3360
Rows remaining for CDR mapping: 0
CDR mapping checkpoint: C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed\checkpoints\cdr_mapping_fixed.csv
CDR mapping success: 3341
CDR mapping failed: 19


,row_index,PDB_ID,INSTANCE,cif_file,success,error,mapping_methods
0,0,pdb_00001ejo,pdb_00001ejo_H_L,C:\Users\yadavalli charishma\Downloads\splits\...,True,NaN,"{""H1"": ""exact"", ""H2"": ""exact"", ""H3"": ""exact"", ..."
1,1,pdb_00001j1p,pdb_00001j1p_H_L,C:\Users\yadavalli charishma\Downloads\splits\...,True,NaN,"{""H1"": ""exact"", ""H2"": ""exact"", ""H3"": ""exact"", ..."
2,2,pdb_00001nbz,pdb_00001nbz_B_A,C:\Users\yadavalli charishma\Downloads\splits\...,True,NaN,"{""H1"": ""exact"", ""H2"": ""exact"", ""H3"": ""exact"", ..."
3,3,pdb_00001dqj,pdb_00001dqj_B_A,C:\Users\yadavalli charishma\Downloads\splits\...,True,NaN,"{""H1"": ""exact"", ""H2"": ""exact"", ""H3"": ""exact"", ..."
4,4,pdb_00001nby,pdb_00001nby_B_A,C:\Users\yadavalli charishma\Downloads\splits\...,True,NaN,"{""H1"": ""exact"", ""H2"": ""exact"", ""H3"": ""exact"", ..."


## 8. Feature matrix helpers

The 5 Å cutoff is between side-chain heavy atoms. Antigen interface residues are identified by contacts to antibody CDR residues. Internal residue contacts are then counted separately within the antibody CDR residue set and antigen interface residue set. Each unordered residue pair is counted once. Contact frequencies are normalized by the total internal contacts for that partner; diagonal same-amino-acid pairs are left zero in the triangular display.

In [9]:
AA_TO_INDEX = {aa: i for i, aa in enumerate(
    ["A","R","N","D","C","Q","E","G","H","I","L","K","M","F","P","S","T","W","Y","V"]
)}

def sidechain_heavy_atoms(residue):
    backbone = {"N", "CA", "C", "O"}
    return [a for a in residue if a.element.upper() not in {"H", "D"} and a.get_name().strip() not in backbone]

def residue_contact(res1, res2, cutoff=5.0):
    atoms1, atoms2 = sidechain_heavy_atoms(res1), sidechain_heavy_atoms(res2)
    return any((a1 - a2) <= cutoff for a1 in atoms1 for a2 in atoms2)

def collect_interface_residues(cdr_residues, antigen_residues, cutoff=5.0):
    return [ag for ag in antigen_residues if any(residue_contact(cdr, ag, cutoff) for cdr in cdr_residues)]

def internal_contact_pairs(residues, cutoff=5.0):
    counts = {}
    # Avoid counting duplicated residues if CDRs overlap.
    unique = {}
    for r in residues:
        key = (r.get_parent().id, r.id)
        unique[key] = r
    residues = list(unique.values())
    for i in range(len(residues)):
        for j in range(i + 1, len(residues)):
            r1, r2 = residues[i], residues[j]
            if not residue_contact(r1, r2, cutoff):
                continue
            aa1 = AA3_TO_1.get(r1.get_resname().upper())
            aa2 = AA3_TO_1.get(r2.get_resname().upper())
            if aa1 not in AA_TO_INDEX or aa2 not in AA_TO_INDEX:
                continue
            pair = tuple(sorted((aa1, aa2), key=lambda aa: AA_TO_INDEX[aa]))
            counts[pair] = counts.get(pair, 0) + 1
    return counts

def frequency_triangle(contact_counts, upper=True):
    matrix = np.zeros((20, 20), dtype=np.float32)
    total = sum(contact_counts.values())
    if total == 0:
        return matrix
    for (aa1, aa2), count in contact_counts.items():
        i, j = AA_TO_INDEX[aa1], AA_TO_INDEX[aa2]
        lo, hi = sorted((i, j))
        # Diagonal excluded; pair-frequency values go into the strict triangle.
        if lo != hi:
            matrix[lo, hi] += count / total if upper else 0
            matrix[hi, lo] += count / total if not upper else 0
    return matrix

def build_one_matrix(row, cif_path, cutoff=5.0):
    structure = parse_cif(cif_path, str(row["PDB_ID"]))
    cdr_map, error, _ = map_all_cdrs(row, structure)
    if cdr_map is None:
        raise ValueError(f"CDR mapping failed: {error}")

    cdr_residues = sum([cdr_map[k] for k in ["H1","H2","H3","L1","L2","L3"]], [])
    antibody_chain_ids = {str(row["Hchain"]).strip().upper(), str(row["Lchain"]).strip().upper()}
    antigen_chain_ids = set(clean_chain_list(row["agchains"]))

    antibody_residues, antigen_residues = [], []
    for model in structure:
        for chain in model:
            cid = str(chain.id).upper()
            target = antibody_residues if cid in antibody_chain_ids else antigen_residues if cid in antigen_chain_ids else None
            if target is not None:
                target.extend([
                    r for r in chain
                    if AA3_TO_1.get(r.get_resname().upper()) in AA_TO_INDEX
                ])

    antigen_interface = collect_interface_residues(cdr_residues, antigen_residues, cutoff)
    antibody_counts = internal_contact_pairs(cdr_residues, cutoff)
    antigen_counts = internal_contact_pairs(antigen_interface, cutoff)

    matrix = frequency_triangle(antibody_counts, upper=False) + frequency_triangle(antigen_counts, upper=True)
    meta = {
        "PDB_ID": row["PDB_ID"], "INSTANCE": row["INSTANCE"],
        "antibody_cdr_residues": len(cdr_residues),
        "antigen_interface_residues": len(antigen_interface),
        "antibody_internal_contacts": sum(antibody_counts.values()),
        "antigen_internal_contacts": sum(antigen_counts.values())
    }
    return matrix, meta

## 9. Build matrices with incremental checkpoints

Only CDR-successful records are eligible. Existing matrix files are reused if they can be loaded and have shape `(20, 20)`. Invalid files are regenerated. Progress is written after every 10 samples and at the end.

In [26]:
status_file = CHECKPOINT_DIR / "feature_status_fixed.csv"
status_cols = ["row_index","PDB_ID","INSTANCE","matrix_file","success","error",
               "antibody_cdr_residues","antigen_interface_residues",
               "antibody_internal_contacts","antigen_internal_contacts"]

if status_file.exists() and status_file.stat().st_size > 0:
    try:
        feature_status = pd.read_csv(status_file)
    except (pd.errors.EmptyDataError, pd.errors.ParserError):
        feature_status = pd.DataFrame(columns=status_cols)
else:
    feature_status = pd.DataFrame(columns=status_cols)

status_by_idx = {}
if not feature_status.empty and "row_index" in feature_status:
    for _, r in feature_status.iterrows():
        try:
            status_by_idx[int(r["row_index"])] = r.to_dict()
        except Exception:
            pass

work = cdr_df[cdr_df["success"].astype(str).str.lower().eq("true")].copy()
print("CDR-successful candidates:", len(work))

for n, (_, info) in enumerate(work.iterrows(), start=1):
    idx = int(info["row_index"])
    matrix_path = MATRIX_DIR / f"matrix_{idx:05d}.npy"

    if matrix_path.exists():
        try:
            arr = np.load(matrix_path)
            if arr.shape == (20, 20) and np.isfinite(arr).all():
                if idx not in status_by_idx:
                    status_by_idx[idx] = {
                        "row_index": idx, "PDB_ID": info["PDB_ID"], "INSTANCE": info["INSTANCE"],
                        "matrix_file": str(matrix_path), "success": True, "error": "",
                        "antibody_cdr_residues": np.nan, "antigen_interface_residues": np.nan,
                        "antibody_internal_contacts": np.nan, "antigen_internal_contacts": np.nan
                    }
                continue
        except Exception:
            pass

    row = filtered.loc[idx]
    try:
        matrix, meta = build_one_matrix(row, info["cif_file"], cutoff=5.0)
        np.save(matrix_path, matrix)
        status_by_idx[idx] = {
            "row_index": idx, "PDB_ID": row["PDB_ID"], "INSTANCE": row["INSTANCE"],
            "matrix_file": str(matrix_path), "success": True, "error": "",
            **meta
        }
    except Exception as exc:
        status_by_idx[idx] = {
            "row_index": idx, "PDB_ID": row["PDB_ID"], "INSTANCE": row["INSTANCE"],
            "matrix_file": "", "success": False, "error": str(exc),
            "antibody_cdr_residues": np.nan, "antigen_interface_residues": np.nan,
            "antibody_internal_contacts": np.nan, "antigen_internal_contacts": np.nan
        }

    if n % 10 == 0 or n == len(work):
        pd.DataFrame(list(status_by_idx.values()), columns=status_cols).sort_values("row_index").to_csv(status_file, index=False)
        print(f"{n}/{len(work)} candidates processed; checkpoint saved")

feature_status = pd.DataFrame(list(status_by_idx.values()), columns=status_cols)
if not feature_status.empty:
    feature_status = feature_status.sort_values("row_index")
feature_status.to_csv(status_file, index=False)

print("Feature status saved:", status_file)
print("Successful matrices:", int(feature_status["success"].astype(str).str.lower().eq("true").sum()) if not feature_status.empty else 0)
print("Failed matrices:", int(feature_status["success"].astype(str).str.lower().eq("false").sum()) if not feature_status.empty else 0)

CDR-successful candidates: 3341
1710/3341 candidates processed; checkpoint saved
1720/3341 candidates processed; checkpoint saved
1730/3341 candidates processed; checkpoint saved
1740/3341 candidates processed; checkpoint saved
1750/3341 candidates processed; checkpoint saved
1760/3341 candidates processed; checkpoint saved
1770/3341 candidates processed; checkpoint saved
1780/3341 candidates processed; checkpoint saved
1790/3341 candidates processed; checkpoint saved
1800/3341 candidates processed; checkpoint saved
1810/3341 candidates processed; checkpoint saved
1820/3341 candidates processed; checkpoint saved
1830/3341 candidates processed; checkpoint saved
1840/3341 candidates processed; checkpoint saved
1850/3341 candidates processed; checkpoint saved
1860/3341 candidates processed; checkpoint saved
1870/3341 candidates processed; checkpoint saved
1880/3341 candidates processed; checkpoint saved
1890/3341 candidates processed; checkpoint saved
1900/3341 candidates processed; check

## 10. Assemble X arrays and report

This only succeeds if at least one valid matrix was generated. Labels for PP/general protein interactions are not created here; this is the ABAG feature-extraction stage.

In [27]:
successful = feature_status[
    feature_status["success"].astype(str).str.lower().eq("true")
].copy() if not feature_status.empty else pd.DataFrame()

if len(successful) == 0:
    print("No matrices have succeeded yet.")
    print("Review chain_verification_fixed.csv and cdr_mapping_fixed.csv for failure reasons.")
else:
    arrays, kept = [], []
    for _, r in successful.sort_values("row_index").iterrows():
        path = Path(r["matrix_file"])
        try:
            arr = np.load(path)
            if arr.shape == (20, 20) and np.isfinite(arr).all():
                arrays.append(arr.astype(np.float32))
                kept.append(r)
        except Exception as exc:
            print("Could not load", path, exc)

    if arrays:
        X = np.stack(arrays)
        X_cnn = X[..., np.newaxis]
        np.save(OUTPUT_DIR / "X_20x20.npy", X)
        np.save(OUTPUT_DIR / "X_cnn_20x20x1.npy", X_cnn)

        kept_df = pd.DataFrame(kept)
        kept_meta = filtered.loc[kept_df["row_index"].astype(int)].copy().reset_index(drop=True)
        kept_meta["matrix_file"] = kept_df["matrix_file"].values
        kept_meta.to_csv(OUTPUT_DIR / "feature_metadata.csv", index=False)

        print("X shape:", X.shape)
        print("CNN input shape:", X_cnn.shape)
        print("Saved arrays and feature_metadata.csv in:", OUTPUT_DIR.resolve())

X shape: (1709, 20, 20)
CNN input shape: (1709, 20, 20, 1)
Saved arrays and feature_metadata.csv in: C:\Users\yadavalli charishma\Desktop\abag_feature_output_fixed


## 11. Diagnostics and summary

Use these tables to see why rows were excluded. If chain matches remain zero, check the local CIF filenames and the required chain IDs shown in `chain_verification_fixed.csv`. This notebook does not silently count failed structures as usable samples.

In [30]:
print("Filtered ABAG rows:", len(filtered))
print("CIF files discovered:", len(cif_paths))
print("Chain verification successes:", int(chain_df["success"].sum()))
print("CDR mapping successes:", int(cdr_df["success"].astype(str).str.lower().eq("true").sum()))
print("Matrix files available:", len(list(MATRIX_DIR.glob("matrix_*.npy"))))

print("\nMost common chain verification errors:")
display(chain_df.loc[~chain_df["success"], "error"].value_counts().head(10))

print("\nMost common CDR mapping errors:")
display(cdr_df.loc[~cdr_df["success"].astype(str).str.lower().eq("true"), "error"].value_counts().head(10))

Filtered ABAG rows: 5432
CIF files discovered: 15589
Chain verification successes: 3360
CDR mapping successes: 3341
Matrix files available: 1709

Most common chain verification errors:


error
No matching local CIF/required chains. pdb_00001e4w_H_L.cif: required ['B/P', 'H', 'L'], found ['B', 'H', 'L', 'P']                                                                                                                                                                  1
No matching local CIF/required chains. pdb_00002igf_H_L.cif: required ['A/P', 'H', 'L'], found ['A', 'H', 'L', 'P']                                                                                                                                                                  1
No matching local CIF/required chains. pdb_00004hkz_B_A.cif: required ['A', 'B', 'C/H'], found ['A', 'B', 'C', 'H']                                                                                                                                                                  1
No matching local CIF/required chains. pdb_00001bj1_H_L.cif: required ['B/V/W', 'J', 'K'], found ['A', 'H', 'L', 'V', 'W'] | pdb_00001bj1_K_J.cif: required [


Most common CDR mapping errors:


error
L1: sequence not mapped (coverage=0.33, identity=1.00)    14
H1: sequence not mapped (coverage=0.25, identity=1.00)     2
H3: sequence not mapped (coverage=0.61, identity=1.00)     1
L1: sequence not mapped (coverage=0.17, identity=1.00)     1
L1: sequence not mapped (coverage=0.67, identity=0.75)     1
Name: count, dtype: int64

In [41]:
import pandas as pd
from pathlib import Path

status_file = CHECKPOINT_DIR / "feature_status_fixed.csv"

feature_status = pd.read_csv(status_file)

print("Columns:", feature_status.columns.tolist())
print("\nTotal records:", len(feature_status))

display(feature_status.head())

print("\nMissing values in each column:")
display(feature_status.isna().sum())

Columns: ['row_index', 'PDB_ID', 'INSTANCE', 'matrix_file', 'success', 'error', 'antibody_cdr_residues', 'antigen_interface_residues', 'antibody_internal_contacts', 'antigen_internal_contacts']

Total records: 3341


,row_index,PDB_ID,INSTANCE,matrix_file,success,error,antibody_cdr_residues,antigen_interface_residues,antibody_internal_contacts,antigen_internal_contacts
0,0,pdb_00001ejo,pdb_00001ejo_H_L,C:\Users\yadavalli charishma\Desktop\abag_feat...,True,NaN,51.0,10.0,68.0,10.0
1,1,pdb_00001j1p,pdb_00001j1p_H_L,C:\Users\yadavalli charishma\Desktop\abag_feat...,True,NaN,40.0,15.0,51.0,12.0
2,2,pdb_00001nbz,pdb_00001nbz_B_A,C:\Users\yadavalli charishma\Desktop\abag_feat...,True,NaN,40.0,15.0,50.0,10.0
3,3,pdb_00001dqj,pdb_00001dqj_B_A,C:\Users\yadavalli charishma\Desktop\abag_feat...,True,NaN,40.0,15.0,50.0,12.0
4,4,pdb_00001nby,pdb_00001nby_B_A,C:\Users\yadavalli charishma\Desktop\abag_feat...,True,NaN,40.0,16.0,51.0,13.0



Missing values in each column:


row_index                        0
PDB_ID                           0
INSTANCE                         0
matrix_file                   1632
success                          0
error                         1709
antibody_cdr_residues         1641
antigen_interface_residues    1641
antibody_internal_contacts    1641
antigen_internal_contacts     1641
dtype: int64

In [42]:
# Separate failed rows
failed = feature_status[
    feature_status["success"].astype(str).str.lower().eq("false")
].copy()

print("Total failed rows:", len(failed))
print("\nMost common failure reasons:")

display(
    failed["error"]
    .fillna("No error recorded")
    .value_counts()
    .head(20)
    .rename_axis("Error")
    .reset_index(name="Count")
)

Total failed rows: 1632

Most common failure reasons:


,Error,Count
0,name 'build_one_matrix' is not defined,1632


In [43]:
display(
    failed[
        ["row_index", "PDB_ID", "INSTANCE", "error"]
    ].head(15)
)

,row_index,PDB_ID,INSTANCE,error
1709,2623,pdb_00008id6,pdb_00008id6_S1_S2,name 'build_one_matrix' is not defined
1710,2624,pdb_00007vuz,pdb_00007vuz_S1_S2,name 'build_one_matrix' is not defined
1711,2625,pdb_00008jgf,pdb_00008jgf_E1_E2,name 'build_one_matrix' is not defined
1712,2626,pdb_00008z3q,pdb_00008z3q_S1_S2,name 'build_one_matrix' is not defined
1713,2627,pdb_00009iv6,pdb_00009iv6_S1_S2,name 'build_one_matrix' is not defined
1714,2628,pdb_00009k3h,pdb_00009k3h_S1_S2,name 'build_one_matrix' is not defined
1715,2629,pdb_00008iw9,pdb_00008iw9_S1_S2,name 'build_one_matrix' is not defined
1716,2630,pdb_00008w8r,pdb_00008w8r_S1_S2,name 'build_one_matrix' is not defined
1717,2631,pdb_00008hcq,pdb_00008hcq_E1_E2,name 'build_one_matrix' is not defined
1718,2632,pdb_00007f4h,pdb_00007f4h_S1_S2,name 'build_one_matrix' is not defined


In [44]:
print("Function exists:", "build_one_matrix" in globals())

if "build_one_matrix" in globals():
    print(build_one_matrix)

Function exists: False
